# TweetEval stance: tokenizer, representaties en modellen (Jeff & Mink)

Project 1: BPE-tokenizer op onze tweets, drie representaties × drie architecturen, vergeleken met cross-validation, macro-F1 en een Bayesiaanse analyse met ROPE.

De herbruikbare code staat in `src/` (`data`, `features`, `models`, `evaluate`); dit notebook roept die aan.

## Setup

In [ ]:
import sys
import warnings
from collections import Counter, OrderedDict
from pathlib import Path

warnings.simplefilter(action="ignore", category=FutureWarning)

# notebook staat in notebooks/, de repo-root is één map hoger
root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1
from src.data import LABEL_NAMES, load_stance, make_cv_splits
from src.evaluate import (
    ROPE, TOKENIZER_MIN_FREQUENCY, TOKENIZER_VOCAB_SIZE,
    compare_systems, plot_comparison, run_cross_validation, summary_table,
)
from src.features import ENCODER_FACTORIES, BagOfWords, TfidfEncoder, TokenIndicator
from src.models import MODEL_RUNNERS

## Data

`#SemST` wordt in `load_stance` (`src/data.py`) verwijderd: artefact uit SemEval-2016, staat in elke tweet en draagt geen informatie.

In [ ]:
stance_dataset = load_stance("feminist")
train_texts = stance_dataset["train"]["text"]
print(f"{len(train_texts)} train-tweets; de eerste:\n{train_texts[0]}")

## BPE stap voor stap

### Pretokenisatie

In [ ]:
prepared_corpus = part1.bpe.prepare_corpus(train_texts)
prepared_corpus[:5]

### Initiële vocabulaire: alle karakters + UNK-teken

In [ ]:
char_vocab = {ch for token in prepared_corpus for ch in token}
vocab = char_vocab | {"\u2047"}  # '⁇' voor karakters die we later nog nooit gezien hebben
sorted(vocab)

### Pretokens tellen

In [ ]:
word_counts = Counter(tuple(word) for word in prepared_corpus)
word_counts.most_common(10)

### Paren tellen en het meest frequente paar kiezen

In [ ]:
pair_counts = Counter()
for symbols, frequency in word_counts.items():
    for left, right in zip(symbols, symbols[1:]):
        pair_counts[(left, right)] += frequency

# bij gelijke telling: lexicografische volgorde als tie-breaker
best_pair = min(pair_counts, key=lambda pair: (-pair_counts[pair], pair))
best_pair, pair_counts[best_pair]

### Eerste merge vastleggen

De volgorde van de merges is wat telt bij het segmenteren van nieuwe tekst.

In [ ]:
merges = OrderedDict()
merges[best_pair] = pair_counts[best_pair]
merges

## Tokenizer trainen

Hugging Face BPE (zoals in de tutorial). Hier op alle train-tweets, alleen om te bekijken hoe tweets opgeknipt worden.
In de experimenten hieronder wordt de tokenizer **per fold** opnieuw getraind op alleen het trainingsdeel, zodat de validatie-fold echt ongezien blijft.

In [ ]:
tokenizer = part1.experiment.train_tokenizer(train_texts, TOKENIZER_VOCAB_SIZE, TOKENIZER_MIN_FREQUENCY)
tokenizer.encode(train_texts[0]).tokens

## Representaties

De klassen staan in `src/features.py`.

- **Token indicators**: 1 als een token in de tweet voorkomt, anders 0.
- **Bag-of-words**: hoe vaak elk token voorkomt.
- **TF-IDF** (Mink): Term Frequency - Inverse Document Frequency. Een token dat vaak in één tweet voorkomt is nuttig, maar minder als het in bijna alle tweets staat. De IDF-gewichten worden alleen op de trainingsdata geschat.

In [ ]:
encoders_demo = {
    "indicator": TokenIndicator(tokenizer),
    "bow": BagOfWords(tokenizer),
    "tfidf": TfidfEncoder(tokenizer, train_texts),
}
n_tokens = len(tokenizer.encode(train_texts[0]).ids)
print(f"tweet 0 heeft {n_tokens} tokens")
for name, encoder in encoders_demo.items():
    vector = encoder(train_texts[0])
    print(f"{name:10s} dim={len(vector)}  som={vector.sum():.2f}  niet-nul={int((vector != 0).sum())}")

## Labelverdeling

Meerderheidsklasse "against" is ~49%: een model dat altijd "against" zegt haalt al 49% accuracy. Daarom gebruiken we **macro-F1**.

In [ ]:
Counter(stance_dataset["train"]["label"])  # 0=none, 1=against, 2=favor

## Experimenten: 3 representaties × 3 architecturen

- **Modellen** (`src/models.py`): log-linear, MLP met 1 verborgen laag, MLP met 2 verborgen lagen.
- Beide MLP's gebruiken **exact dezelfde training** (zelfde optimizer, dropout, epochs, keuze van het beste epoch, seed). Het enige verschil is de extra laag, dus een verschil in score komt ook echt door die laag.
- Per fold worden tokenizer, TF-IDF en model alleen op het trainingsdeel gefit (`src/evaluate.py`).

In [ ]:
cv_runs = make_cv_splits(stance_dataset)
f1_scores = run_cross_validation(cv_runs, ENCODER_FACTORIES, MODEL_RUNNERS, num_classes=len(LABEL_NAMES))

### Gemiddelde macro-F1

In [ ]:
encoder_names = list(ENCODER_FACTORIES)
model_names = list(MODEL_RUNNERS)
results_table = summary_table(f1_scores, encoder_names, model_names)
results_table.round(3)

## Bayesiaanse vergelijking met ROPE

Correlated t-test van Benavoli et al. (via `part1.analysis.BenavoliCorrelatedT`), op de macro-F1 per fold.

**ROPE = 0.01**: een verschil van minder dan 1 procentpunt macro-F1 noemen we praktisch gelijk.
TODO (samen beslissen): motivatie aanscherpen. Bijvoorbeeld: een validatie-fold heeft ~120 tweets, dus één tweet anders voorspellen verschuift de score al met ongeveer 1 procentpunt; kleinere verschillen zijn ruis.

### Architecturen vergelijken (met de beste representatie)

In [ ]:
n_runs = len(cv_runs)
best_encoder = results_table.max(axis=0).idxmax()
architectures = [f"{model}_{best_encoder}" for model in model_names]
print("beste representatie:", best_encoder)
compare_systems(f1_scores, architectures, runs=n_runs)

In [ ]:
for i, sys_a in enumerate(architectures):
    for sys_b in architectures[i + 1:]:
        plot_comparison(f1_scores, sys_a, sys_b, runs=n_runs)

### Representaties vergelijken (met het beste model)

In [ ]:
best_model = results_table.max(axis=1).idxmax()
representations = [f"{best_model}_{enc}" for enc in encoder_names]
print("beste model:", best_model)
compare_systems(f1_scores, representations, runs=n_runs)

In [ ]:
for i, sys_a in enumerate(representations):
    for sys_b in representations[i + 1:]:
        plot_comparison(f1_scores, sys_a, sys_b, runs=n_runs)